# Aula 20 — Agentic Systems Foundations

**Quando a próxima ação deixa de ser fixa?**

Na Aula 18, aprendemos workflows determinísticos.  
Na Aula 19, aprendemos a expor e descobrir capabilities via MCP.

Agora entra uma nova variável:

> **autonomia na escolha da próxima ação.**

Regra da aula:

```text
não chamar tudo de agente
→ tornar a decisão observável
→ comparar com alternativa mais simples
→ medir benefício, custo e risco
```

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir workflow determinístico de agentic loop;
- localizar onde a autonomia aparece;
- separar decisão, autorização e execução;
- implementar estado, action space, step budget e termination conditions;
- observar human escalation;
- comparar workflow e agentic loop sob os mesmos cenários;
- justificar quando autonomia adicional não compensa.

## 📘 Glossário da aula

Conceitos centrais:

- **[Loop agente](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#loop-agente)**
- **[Autonomia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#autonomia)**
- **[Orçamento de passos](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orçamento-de-passos)**
- **[Condição de término](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#condição-de-término)**
- **[Escalonamento humano](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#escalonamento-humano)**

Use o glossário durante o laboratório, não apenas como referência final.

## 2. Tool, workflow, MCP e agente não são sinônimos

```text
Tool
→ executa uma capacidade

Workflow
→ coordena capacidades sob transições explícitas

MCP
→ padroniza exposição e discovery

Agentic loop
→ observa estado
→ escolhe próxima ação
→ executa sob controles
→ observa resultado
→ decide novamente
```

Um workflow pode ter `if`, branches, retries e checkpoints e ainda continuar sendo determinístico.

A pergunta decisiva é:

> **Quem ou o que escolhe a próxima ação a partir do estado observado?**

## 3. Arquitetura de referência

```text
Goal
 ↓
Observed State
 ↓
Decision Provider
 ↓
Proposed Action
 ↓
Governance Gate
 ↓
Execution
 ↓
Observation
 ↓
State Update
 ↺
```

Importante:

```text
decisão ≠ autorização
decisão ≠ execução
agentic loop ≠ loop ilimitado
```

Nesta aula, o decision provider será determinístico e local. Isso nos permite estudar **agência sem confundir agência com LLM**.

In [ ]:
from dataclasses import dataclass, field, asdict
from copy import deepcopy
import pandas as pd

ACTIONS = {
    "inspect_request",
    "get_lesson_status",
    "build_release_note",
    "request_approval",
    "publish_simulated",
    "ask_human",
    "stop",
}

SCENARIOS = {
    "ready_approved": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": True,
    },
    "not_ready": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "review-candidate",
        "approval_available": True,
    },
    "ambiguous": {
        "goal": "faça o que for melhor com a aula",
        "request_valid": None,
        "lesson_status": "ready",
        "approval_available": True,
    },
    "approval_missing": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": False,
    },
}

print("Cenários:", list(SCENARIOS))
print("Action space:", sorted(ACTIONS))

## 4. Estado explícito

O agente não deve depender de “memória invisível”.

O estado carregará apenas informação operacional necessária para a decisão.

In [ ]:
@dataclass
class AgentState:
    goal: str
    request_valid: object = None
    lesson_status: object = None
    approval_available: bool = False
    inspected: bool = False
    release_note: object = None
    approval: object = None
    published: bool = False
    needs_human: bool = False
    done: bool = False
    step_count: int = 0
    history: list = field(default_factory=list)
    termination_reason: object = None

def initial_state(scenario):
    # Fair comparison: lesson_status is NOT preloaded.
    # Both architectures must obtain it through the same capability.
    return AgentState(
        goal=scenario["goal"],
        request_valid=scenario["request_valid"],
        lesson_status=None,
        approval_available=scenario["approval_available"],
    )

## 5. Capabilities locais

As capabilities são deliberadamente simples. Nenhuma produz side effect externo real.

In [ ]:
def execute_action(action, state, scenario):
    if action == "inspect_request":
        state.inspected = True
        return {"request_valid": state.request_valid}

    if action == "get_lesson_status":
        state.lesson_status = scenario["lesson_status"]
        return {"lesson_status": state.lesson_status}

    if action == "build_release_note":
        note = f"Release simulada — status={state.lesson_status}"
        state.release_note = note
        return {"release_note": note}

    if action == "request_approval":
        state.approval = bool(state.approval_available)
        return {"approval": state.approval}

    if action == "publish_simulated":
        state.published = True
        return {"published": True}

    if action == "ask_human":
        state.needs_human = True
        state.done = True
        state.termination_reason = "human_escalation"
        return {"needs_human": True}

    if action == "stop":
        state.done = True
        state.termination_reason = state.termination_reason or "safe_stop"
        return {"done": True}

    raise ValueError(f"Ação desconhecida: {action}")

## 6. Baseline — workflow determinístico

Primeiro, resolvemos o problema com uma sequência conhecida.

Esse baseline é importante: **não podemos justificar autonomia sem comparar com algo mais simples**.

In [ ]:
def run_workflow(scenario):
    state = initial_state(scenario)
    trace = []
    sequence = [
        "inspect_request",
        "get_lesson_status",
        "build_release_note",
        "request_approval",
        "publish_simulated",
    ]

    for step, action in enumerate(sequence, start=1):
        if action == "inspect_request" and state.request_valid is not True:
            state.needs_human = True
            state.done = True
            state.termination_reason = "human_escalation"
            break

        if action == "build_release_note" and state.lesson_status != "ready":
            state.done = True
            state.termination_reason = "lesson_not_ready"
            break

        if action == "publish_simulated" and state.approval is not True:
            state.needs_human = True
            state.done = True
            state.termination_reason = "human_escalation"
            break

        result = execute_action(action, state, scenario)
        trace.append({"step": step, "action": action, "result": result})

    if state.published:
        state.done = True
        state.termination_reason = "goal_completed"

    return state, trace

wf_state, wf_trace = run_workflow(SCENARIOS["ready_approved"])
display(pd.DataFrame(wf_trace))
print("Termination:", wf_state.termination_reason)

## 7. Decision Provider — onde a agência entra

Agora a próxima ação não vem de uma lista fixa.

O provider recebe o estado atual e propõe **uma ação dentre as permitidas**.

Ele é determinístico nesta aula para tornar a lógica auditável.

In [ ]:
def choose_next_action(state):
    if state.done:
        return "stop", "estado já terminal"

    if state.request_valid is None:
        return "ask_human", "solicitação ambígua"

    if state.request_valid is False:
        return "stop", "solicitação inválida"

    if not state.inspected:
        return "inspect_request", "pedido ainda não foi inspecionado"

    if state.lesson_status is None:
        return "get_lesson_status", "status da aula ainda não foi observado"

    if state.lesson_status != "ready":
        return "stop", "aula ainda não está pronta"

    if state.release_note is None:
        return "build_release_note", "release note ainda não existe"

    if state.approval is None:
        return "request_approval", "publicação exige aprovação"

    if state.approval is False:
        return "ask_human", "aprovação não disponível"

    if not state.published:
        return "publish_simulated", "pré-condições atendidas"

    return "stop", "objetivo concluído"

## 8. Governance Gate

A proposta do decision provider não executa automaticamente.

Antes de executar, verificamos:

- action allow-list;
- step budget;
- política para publicação;
- estado incompatível.

Isso preserva:

> **decisão do agente não é autorização.**

In [ ]:
def governance_gate(action, state, max_steps):
    if action not in ACTIONS:
        return False, "action_not_allowed"

    if state.step_count >= max_steps:
        return False, "step_budget_exceeded"

    if action == "publish_simulated" and state.approval is not True:
        return False, "publish_requires_approval"

    return True, "allowed"

## 9. Agentic loop mínimo

Observe o ciclo:

```text
estado
→ decisão
→ gate
→ execução
→ observação
→ novo estado
```

O histórico registra **ações e razões resumidas**, não raciocínio privado do modelo.

In [ ]:
def run_agentic(scenario, max_steps=7):
    state = initial_state(scenario)
    trace = []

    while not state.done:
        action, reason = choose_next_action(state)
        allowed, gate_reason = governance_gate(action, state, max_steps)

        before = {
            "step_count": state.step_count,
            "lesson_status": state.lesson_status,
            "published": state.published,
            "needs_human": state.needs_human,
            "approval": state.approval,
        }

        if not allowed:
            state.done = True
            state.termination_reason = gate_reason
            trace.append({
                "step": state.step_count + 1,
                "action": action,
                "decision_reason": reason,
                "governance": gate_reason,
                "result": None,
                "state_before": before,
            })
            break

        state.step_count += 1
        result = execute_action(action, state, scenario)

        if action == "stop" and state.termination_reason == "safe_stop":
            if state.lesson_status != "ready":
                state.termination_reason = "lesson_not_ready"
            elif state.published:
                state.termination_reason = "goal_completed"

        if state.published:
            state.done = True
            state.termination_reason = "goal_completed"

        trace.append({
            "step": state.step_count,
            "action": action,
            "decision_reason": reason,
            "governance": gate_reason,
            "result": result,
            "state_before": before,
        })

    return state, trace

agent_state, agent_trace = run_agentic(SCENARIOS["ready_approved"])
display(pd.DataFrame(agent_trace))
print("Termination:", agent_state.termination_reason)

## 10. Comparative Evidence Lab

Agora comparamos as duas arquiteturas sobre os **mesmos quatro cenários**.

A pergunta não é “qual parece mais inteligente?”.

A pergunta é:

> **Que comportamento adicional a autonomia produziu e qual custo operacional veio junto?**

In [ ]:
def summarize_run(architecture, scenario_name, state, trace):
    actions = [row["action"] for row in trace]
    return {
        "architecture": architecture,
        "scenario": scenario_name,
        "success": bool(state.published),
        "steps": len(trace),
        "decision_count": len(trace) if architecture == "agentic_loop" else 0,
        "tool_calls": sum(a not in {"stop", "ask_human"} for a in actions),
        "human_intervention": bool(state.needs_human),
        "termination_reason": state.termination_reason,
        "cost_proxy": len(trace) + (0.5 * (len(trace) if architecture == "agentic_loop" else 0)),
        "latency_proxy": len(trace),
    }

rows = []

for name, scenario in SCENARIOS.items():
    wf_state, wf_trace = run_workflow(scenario)
    rows.append(summarize_run("workflow", name, wf_state, wf_trace))

    ag_state, ag_trace = run_agentic(scenario)
    rows.append(summarize_run("agentic_loop", name, ag_state, ag_trace))

comparison = pd.DataFrame(rows)
display(comparison)

### Como interpretar

A comparação é deliberadamente justa:

```text
mesmo objetivo
+ mesmos cenários
+ mesmas capabilities
+ mesma informação inicial
```

Em particular, nenhuma arquitetura recebe o `lesson_status` gratuitamente: ambas precisam obtê-lo por `get_lesson_status`.

Procure três tipos de evidência:

1. **mesmo resultado, mais complexidade** — autonomia não se justificou naquele cenário;
2. **menos trabalho desnecessário ou adaptação ao estado** — a seleção dinâmica de ações trouxe benefício;
3. **novo ponto de falha** — decisão, loop ou policy criaram risco que o workflow não tinha.

Não existe obrigação de o agentic loop vencer.

> **Se o workflow resolver igualmente bem com menor custo e risco, ele é a melhor arquitetura.**

## 11. Failure Lab — step budget

Um agente precisa saber parar.

Vamos criar deliberadamente um provider defeituoso que nunca progride.

In [ ]:
def run_broken_loop(max_steps=3):
    scenario = SCENARIOS["ready_approved"]
    state = initial_state(scenario)
    trace = []

    while not state.done:
        action = "inspect_request"  # provider defeituoso: repete a mesma ação
        allowed, gate_reason = governance_gate(action, state, max_steps)

        if not allowed:
            state.done = True
            state.termination_reason = gate_reason
            break

        state.step_count += 1
        result = execute_action(action, state, scenario)
        trace.append({"step": state.step_count, "action": action, "result": result})

    return state, trace

broken_state, broken_trace = run_broken_loop()
display(pd.DataFrame(broken_trace))
print("Termination:", broken_state.termination_reason)

### Interpretação

O `step_budget_exceeded` não é um detalhe de implementação.

Ele é evidência de uma diferença arquitetural:

```text
workflow
→ número de transições conhecido pela estrutura

agentic loop
→ precisa de controles explícitos para evitar não-terminação
```

A autonomia cria capacidade — e também cria uma superfície adicional de governança.

## 12. Architecture Decision Lab

Escolha a arquitetura mínima suficiente:

| Caso | Situação |
|---|---|
| A | sequência estável, baixa variabilidade, alto risco |
| B | poucas branches conhecidas e auditáveis |
| C | objetivo claro, mas ordem das ações depende do estado observado |
| D | tarefa ambígua e irreversível |

Escolha entre:

```text
direct_function
deterministic_workflow
agentic_loop
```

Justifique usando:

- variabilidade;
- previsibilidade;
- reversibilidade;
- custo;
- risco;
- necessidade de supervisão.

> Não escolha `agentic_loop` apenas porque é a opção mais nova.

In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q20_arch = TILExercise(
    "Pergunte primeiro se a ordem das ações precisa ser escolhida em runtime. Depois considere risco e reversibilidade.",
    "A tende a deterministic_workflow; B também tende a deterministic_workflow; C é o melhor candidato a agentic_loop; D deve priorizar supervisão humana e pode nem justificar autonomia.",
)

print("Use q20_arch.hint() ou q20_arch.solution() somente quando desejar.")

In [ ]:
# Registre sua decisão antes de revelar a solução.
my_decisions = {
    "A": None,
    "B": None,
    "C": None,
    "D": None,
}

my_decisions

In [ ]:
# Descomente somente depois de tentar.
# q20_arch.solution()

## 13. Checagem de compreensão

Responda sem olhar o código:

1. Por que um workflow com branches não é automaticamente um agente?
2. Onde a autonomia aparece no laboratório?
3. Por que `choose_next_action()` não deve executar diretamente a tool?
4. O que o governance gate protege?
5. Por que um step budget é parte da arquitetura e não apenas “defensive programming”?
6. Quando `ask_human` é um resultado melhor que continuar executando?
7. Em que cenário o workflow foi suficiente e a autonomia apenas adicionou custo?

## 14. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
decision provider determinístico
action space fechado
step budget explícito
```

O objetivo desta primeira versão é isolar o mecanismo de agência antes de adicionar modelos generativos ou frameworks.

## 15. Síntese

A cadeia da aula foi:

```text
capabilities
→ workflow determinístico
→ estado explícito
→ decision provider
→ governance gate
→ agentic loop
→ observação
→ termination / human escalation
→ comparação de utility
```

### Ideia principal

> **Agência não é possuir tools. É introduzir autonomia controlada na escolha da próxima ação a partir do estado observado.**

E a regra do TIL continua valendo:

> **Se um workflow resolve o problema igualmente bem com menor risco e complexidade, prefira o workflow.**

---

## Continue no TIL

← **[Anterior: Aula 19 — Model Context Protocol (MCP)](https://www.kaggle.com/code/pedrogentil/til-19-model-context-protocol)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **Próxima unidade: Agentic Systems — em preparação** →